LinearRegression model using CSV

In [5]:
from sklearn.metrics import r2_score

def add_bias_column(X):
    """
    Args:
        X (array): can be either 1-d or 2-d
    
    Returns:
        Xnew (array): the same array, but 2-d with a column of 1's in the first spot
    """
    
    # If the array is 1-d
    if len(X.shape) == 1:
        Xnew = np.column_stack([np.ones(X.shape[0]), X])
    
    # If the array is 2-d
    elif len(X.shape) == 2:
        bias_col = np.ones((X.shape[0], 1))
        Xnew = np.hstack([bias_col, X])
        
    else:
        raise ValueError("Input array must be either 1-d or 2-d")

    return Xnew

In [6]:
def line_of_best_fit(X, y):
    """
    Computes the coefficients of the line of best fit for the given predictors and response values.

    Args:
        X (array): Predictor values 
        y (array): Response values
    
    Returns:
        m (tuple): Coefficients for the line of best fit, including the intercept term.
    """
    # Add bias column to X
    X = add_bias_column(X)
    
    # Compute beta using the normal equation
    m = np.linalg.inv(X.T @ X) @ X.T @ y
    
    return m

In [7]:
def linreg_predict(Xnew, ynew, m):
    """
    Args:
        Xnew (array): can be either 1-d or 2-d array, holds predictor values
        ynew (array): 1-d array, target values
        m (array): 1-d array, contains line of best fit coefficients
    
    Returns:
        pred_dict (dictionary): a dictionary with keys and values for the pred y values, the model's resids,  the mse, and  r2 scores
    """
    
    X = add_bias_column(Xnew)
    pred_dict = {
    "ypreds": None,
    "resids": None,
    "mse": 0,
    "r2": 0
    }

    ypreds = np.matmul(X, m)
    resids = ynew-ypreds
    mse = (resids**2).mean()
    r2 = r2_score(ynew, ypreds)

    # fill the dict with values
    pred_dict["ypreds"] = ypreds
    pred_dict["resids"] = resids
    pred_dict["r2"] = r2
    pred_dict["mse"] = mse
    
    
    return pred_dict
    

In [28]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split


# Replace 'your_file.csv' with the path to your CSV file
file_path = 'movies_output_clean.csv'

# Load the CSV file into a Pandas DataFrame
df = pd.read_csv(file_path)

df_first = df[['Genre', 'Rotten Tomatoes Rating', 'BoxOffice', 'IMDb rating']]

df_drops = df_first.dropna()
# After dropping instances of we are left with 732 movies


In [32]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Data Preprocessing
df_x = df_drops[['Genre', 'Rotten Tomatoes Rating', 'BoxOffice']].copy()
df_x['Rotten Tomatoes Rating'] = df_x['Rotten Tomatoes Rating'].str.rstrip('%').astype(float)
df_x['BoxOffice'] = df_x['BoxOffice'].replace(r'[\$,]', '', regex=True).astype(float)
df_x['Genre'] = df_x['Genre'].str.split(',').str[0].str.strip()

# One-hot encoding
df_x_encoded = pd.get_dummies(df_x, columns=['Genre'], prefix='Genre')

# Target variable
y = df_drops['IMDb rating']

# Feature scaling
scaler = StandardScaler()
X = scaler.fit_transform(df_x_encoded)

# Train-test split
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.3, random_state=3)

# Model Training
model = LinearRegression()
model.fit(Xtrain, ytrain)

# Predictions
y_pred = model.predict(Xtest)

# Evaluation
mse = mean_squared_error(ytest, y_pred)
r2 = r2_score(ytest, y_pred)

print(f"MSE: {mse}")
print(f"R²: {r2}")


MSE: 0.34120910374359437
R²: 0.6436519192498074
